# Random sample of AF2 structures from rBMRB (train + val + test)

Draws `N_PICK` proteins with a fixed `SEED`, copies their PDB files to `OUT_DIR` and writes `manifest.csv`.

In [ ]:
import shutil
from pathlib import Path

import numpy as np
import pandas as pd

STRUCT_ROOT = Path("../../../data/rBMRB/cluster_30p/structures_split_filtered")
SPLITS = ["train", "val", "test"]
N_PICK = 20
SEED = 0
OUT_DIR = Path("random20_af2_structures")      # relative to this notebook; change as needed

pool = pd.DataFrame([dict(pid=p.stem, split=s, src=str(p))
                     for s in SPLITS for p in sorted((STRUCT_ROOT / s).glob("*.pdb"))])
if pool.pid.duplicated().any():
    raise ValueError(f"pid in more than one split: {sorted(pool.pid[pool.pid.duplicated()])}")
print(pool.split.value_counts().reindex(SPLITS).to_string(), f"\ntotal {len(pool)} structures")

In [ ]:
rng = np.random.default_rng(SEED)
pick = pool.iloc[np.sort(rng.choice(len(pool), N_PICK, replace=False))].reset_index(drop=True)


def n_residues(path):
    """Number of CA atoms in the first model."""
    n = 0
    for line in open(path):
        if line.startswith("ENDMDL"):
            break
        if line.startswith("ATOM") and line[12:16].strip() == "CA":
            n += 1
    return n


OUT_DIR.mkdir(parents=True, exist_ok=True)
existing = sorted(p.name for p in OUT_DIR.glob("*.pdb"))
if existing:
    raise FileExistsError(f"{OUT_DIR} already has {len(existing)} PDB files -- use an empty folder or delete them")
for r in pick.itertuples():
    shutil.copy2(r.src, OUT_DIR / f"{r.pid}.pdb")
pick["n_res"] = [n_residues(p) for p in pick.src]
pick.to_csv(OUT_DIR / "manifest.csv", index=False)
print(f"copied {len(pick)} structures to {OUT_DIR.resolve()}")
print(pick[["pid", "split", "n_res"]].to_string(index=False))
print(pick.split.value_counts().reindex(SPLITS, fill_value=0).to_string())